# Math 372 · Computational Lab 2
## The dictionary by machine

**Assigned** Wednesday, 4 November · **Due** Friday, 20 November, 10 p.m., via Gradescope

---

The big idea, and its foil:

> **A specification is an algorithm: write it down and the power series does the counting.
> But a computed series is only as good as what you checked it against.**

You will turn sentences ("a set of cycles", "a sequence of nonempty sets", "a root with a set of
subtrees attached") into power series, read counts off them, and check every count against
something that did not produce it: a brute-force enumeration, a theorem from Act I, or the OEIS.
Along the way three things will go wrong on purpose. Each one is a fact about power series worth
knowing.

### How it works (the same as Lab 1)

- Four cells are marked **YOUR TURN**. Each takes a few lines, and the cell just above it shows
  you the pattern. Everything else is given: run it and read it.
- Every YOUR TURN cell has a **prediction** cell before it. Fill the prediction in *before* you
  run anything, and leave it visible if it turns out wrong. A wrong prediction you left visible is
  worth more than a right one you back-filled.
- The YOUR TURN cells end with `raise NotImplementedError(...)`. Delete that line once the cell is
  filled in. **When Restart & Run All completes without error, you are done.**
- This is built to take about **three hours**. If you pass five, stop, write up what you have, and
  email me.

If a cell you were *given* fails, that is my problem, not yours. Email me.

---
## Setup

Run this first. If it fails, you are on the wrong kernel: **Kernel → Change kernel → SageMath**.

We work in $\mathbb Q[[z]]$, keeping the coefficients $a_0,\dots,a_{N-1}$. Sage prints a series
with an `O(z^N)` at the end: everything before it is exact, and nothing after it is known.

In [ ]:
try:
    PowerSeriesRing
except NameError:
    raise RuntimeError("Not a Sage kernel. Kernel > Change kernel > SageMath.")

N = 30                                        # keep a_0, ..., a_{N-1}
R.<z> = PowerSeriesRing(QQ, default_prec=N)

def coeffs(F, n=12):
    "The first n coefficients of F, as a list."
    return [F[k] for k in range(n)]

def labelled(F, n=12):
    "The first n counts of a LABELLED class whose EGF is F: the numbers k! [z^k] F."
    return [factorial(k) * F[k] for k in range(n)]

def check(name, computed, brute):
    "Compare two lists of counts and say whether they agree."
    computed, brute = list(computed), list(brute)
    ok = computed == brute
    print(("AGREE     " if ok else "DISAGREE  ") + name)
    if not ok:
        print("   computed:", computed)
        print("   brute   :", brute)
    return ok

# The unlabelled constructions (as in the October 14 notebook). The class must have no object of size 0.
def SEQ(A):
    assert A.valuation() > 0, "SEQ(A) needs A(0) = 0"
    return 1 / (1 - A)

def MSET(A):
    assert A.valuation() > 0, "MSET(A) needs A(0) = 0"
    return sum(A(z^k) / k for k in range(1, N)).exp()

def PSET(A):
    assert A.valuation() > 0, "PSET(A) needs A(0) = 0"
    return sum((-1)^(k - 1) * A(z^k) / k for k in range(1, N)).exp()

# The labelled constructions (November 2 and 4). Same rule: no object of size 0.
def LSEQ(A):
    assert A.valuation() > 0, "LSEQ(A) needs A(0) = 0"
    return 1 / (1 - A)

def SET(A):
    assert A.valuation() > 0, "SET(A) needs A(0) = 0"
    return A.exp()

def CYC(A):
    assert A.valuation() > 0, "CYC(A) needs A(0) = 0"
    return (1 / (1 - A)).log()

print("Setup OK.")

---
# Part 0 · What the O-term means

A series Sage has computed is a finite list of exact numbers plus a promise: the `O(z^n)` marks
where the list stops being known. Here is the Fibonacci series, cut off on purpose at $z^{10}$.

In [ ]:
G = (1 / (1 - z - z^2)).add_bigoh(10)
G

### Prediction 0.1

Before running the next cell, say what each line will print, **including where the O-term lands**.

- (a) `G * (1 - z - z^2)`
- (b) `(G - 1) / z`
- (c) `G(z^3)`, the series with $z^3$ substituted for $z$
- (d) `G * G(z^2)`, a product of a series known to $O(z^{10})$ and one known to $O(z^{20})$

*Your predictions:*

In [ ]:
print("(a)", G * (1 - z - z^2))
print("(b)", (G - 1) / z)
print("(c)", G(z^3))

**Question 0.1.** In one sentence each, say why the O-term moved (or didn't) in (a), (b) and (c).
Which of the operations *gained* information about high coefficients, and how is that
possible? And in (d), what decides where the O-term lands?

*Your answer:*

In [ ]:
print("(d)", G * G(z^2), "   precision:", (G * G(z^2)).prec())

---
# Part 1 · Unlabelled classes

### 1.1 · The pattern

Partitions are multisets of positive integers: $\mathcal P = \operatorname{MSET}(\mathcal I)$
with $I(z) = z/(1-z)$. Here is the full pattern every YOUR TURN in this lab follows:
**specification → series → counts → check against brute force**.

In [ ]:
I = z / (1 - z)                                   # one positive integer of each size
P = MSET(I)
print(coeffs(P, 15))

brute = [len(list(Partitions(n))) for n in range(15)]  # list them all and count, n < 15
check("partitions, MSET vs listing", coeffs(P, 15), brute)

### Prediction 1.2

A multiset is a "set with repetition allowed," and $\exp$ is the function that turns a series into
the series of its *sets*. So here is a tempting shortcut: $\operatorname{MSET}(\mathcal A)$ should be
roughly $\exp(A(z))$. **What do you expect the coefficients of $\exp(I(z))$ to be?** Write down the
first four.

*Your prediction:*

In [ ]:
print(coeffs(I.exp(), 8))

**Question 1.2.** These coefficients are not even integers, so $\exp(I)$ is not the generating
function of any class at all, in the sense we have been using. What went wrong with the shortcut?
Your answer should mention what the terms $A(z^2), A(z^3), \dots$ in the `MSET` formula are doing.

Then try `labelled(I.exp(), 8)`. The numbers it prints *are* counts. You will see what they count in
Part 2; for now, say what you notice.

*Your answer:*

(For a second unlabelled check with real content, `PSET` of the odd integers, `z/(1-z^2)`, counts
partitions into distinct odd parts, and these are equinumerous with the self-conjugate partitions;
`p == p.conjugate()` tests the latter. Not required.)

---
# Part 2 · Labelled classes

A labelled class has an **exponential** generating function, $\sum a_n z^n/n!$, so the counts are
$n!\,[z^n]F$. That is what `labelled(F)` prints. The constructions are `LSEQ`, `SET` and `CYC`,
and they translate to $\frac1{1-A}$, $e^{A}$ and $\log\frac1{1-A}$.

### 2.1 · The pattern, twice

**Permutations** are sets of cycles. **Set partitions** are sets of nonempty sets.

In [ ]:
Zl = z                                            # the labelled atom: EGF z
perms = SET(CYC(Zl))
print("permutations:", labelled(perms, 10))
check("permutations vs n!", labelled(perms, 10), [factorial(n) for n in range(10)])

nonempty_set = Zl.exp() - 1                       # SET_{>=1}(Z)
bell = SET(nonempty_set)
print("set partitions:", labelled(bell, 10))
check("set partitions vs listing", labelled(bell, 9), [len(list(SetPartitions(n))) for n in range(9)])

### Prediction 2.2

Two labelled classes, built from the same two pieces in opposite orders:

- $\mathcal U = \operatorname{SET}(\operatorname{LSEQ}_{\ge1}(\mathcal Z))$: **sets of nonempty lists**;
- $\mathcal V = \operatorname{LSEQ}(\operatorname{SET}_{\ge1}(\mathcal Z))$: **lists of nonempty sets**
  (also called ordered set partitions).

Count both by hand for $n = 0, 1, 2, 3$. Then predict: are $\mathcal U$ and $\mathcal V$ equinumerous?

*Your counts and prediction:*

In [ ]:
U = SET(Zl / (1 - Zl))
V = LSEQ(nonempty_set)
print("sets of lists :", labelled(U, 9))
print("lists of sets :", labelled(V, 9))

**Question 2.2.** Where do the two sequences first differ? What does this tell you about
recognizing a sequence from its first few terms, for instance by typing them into the OEIS? Look
up both sequences there and record their A-numbers.

*Your answer:*

### 2.3 · Your turn: derangements

A **derangement** is a permutation with no fixed points: a set of cycles, **each of length at
least 2**.

### Prediction 2.3

(a) Write the EGF as a closed formula before computing anything. (b) The probability that a
random permutation of $[n]$ is a derangement is $D_n/n!$. What does it tend to, and roughly how
close is it by $n = 10$?

*Your predictions:*

In [ ]:
# YOUR TURN
# 1. CYC_{>=2}(Z): the cycles of length at least 2. (What do you subtract from CYC(Zl)?)
# 2. der = SET(...); print labelled(der, 12).
# 3. Brute force for n < 9: count permutations with no fixed points.
#    In Sage, p.number_of_fixed_points() works for p in Permutations(n).
# 4. check(...), then print D_n / n! as a decimal for n = 2, 4, ..., 14 next to 1/e.


raise NotImplementedError("delete this line when the cell is filled in")

---
# Part 3 · Recursion: trees

A tree is defined in terms of trees, so its generating function is defined by an *equation*. One
way to solve such an equation is to iterate it. For plane trees, $G = z/(1-G)$: start from
$G_0 = 0$ and repeat $G_{k+1} = z/(1-G_k)$.

**One rule for iterating:** start from `O(z^N)`, not from the exact `0`. An exact polynomial has
no O-term, so it is never truncated, and its degree can double at every step until the kernel
runs out of memory.

### Prediction 3.1

After $k$ steps, how many coefficients of $G_k$ agree with the true $G$? And what will Sage's
`.prec()` say about $G_k$?

*Your prediction:*

In [ ]:
true_G = (1 - (1 - 4*z).sqrt()) / 2               # the quadratic formula, from October 30
Gk = O(z^N)                                       # "0, known to O(z^N)"
for k in range(1, 6):
    Gk = z / (1 - Gk)
    first_wrong = next((i for i in range(N) if Gk[i] != true_G[i]), None)
    print("k = %d   prec() = %d   first wrong coefficient: z^%s   %s" % (k, Gk.prec(), first_wrong, coeffs(Gk, 8)))

**Question 3.1.** Sage reports a precision of about $N+k$, but only the coefficients through $z^k$
are right. Both statements are true. Explain what each one measures. (Part 0 is the place to
start.) How many steps would you need for all $N$ coefficients to be right?

*Your answer:*

### 3.2 · Your turn: labelled trees, and a callback to Act I

A **rooted labelled tree** is a root (a labelled atom) with a *set* of rooted labelled subtrees:
$\mathcal T = \mathcal Z \star \operatorname{SET}(\mathcal T)$, so $T(z) = z\,e^{T(z)}$.

### Prediction 3.2

Lagrange inversion (October 30) gives $n!\,[z^n]\,T = n^{n-1}$. Taking the root away, the number of
*unrooted* labelled trees on $[n]$ should be $n^{n-2}$. Where in Act I did you meet $n^{n-2}$, and
what did it count there?

*Your prediction:*

In [ ]:
# YOUR TURN
# 1. Iterate T = z * exp(T), starting from O(z^N), enough times that all N coefficients are right.
# 2. check(...) labelled(T, 12) against [n^(n-1) for n in range(12)]   (careful at n = 0).
# 3. Act I: graphs.CompleteGraph(n).spanning_trees_count() is the Matrix-Tree count kappa(K_n).
#    check(...) it against labelled(T)[n] / n for n = 1, ..., 9.


raise NotImplementedError("delete this line when the cell is filled in")

---
# Part 4 · A class of your own

Pick one class.

1. **Involutions**: permutations whose cycles all have length $1$ or $2$. (Brute force: in Sage,
   `p.order() <= 2` for `p in Permutations(n)`.)
2. **Set partitions with no singleton blocks**. (Brute force: filter `SetPartitions(n)` by block
   sizes.)
3. **Your own**, from a project, from the course, or from somewhere else, as long as you can write
   the specification and list the small objects by brute force without writing a program of your
   own; the two options above need one line each.

The two named options are labelled, so the counts are $n!\,[z^n]F$, and both have a surprise in
the growth question. For the ratio test, the October 30 notebook on the course page has the model:
its last cell measures the $n^{-3/2}$ in the Catalan numbers.

### Prediction 4.1 · three predictions, before computing anything

1. The first five counts, **by hand**, with the objects listed for at least one $n\ge3$.
2. Labelled or unlabelled, and so: ordinary or exponential generating function?
3. How fast will the counts grow: like $\rho^{-n}$ for which $\rho$ (and why), or faster than any
   exponential?

*Your three predictions, with reasons:*

1.
2.
3.

In [ ]:
# YOUR TURN
# 1. Write the specification as a series, called F. (For an equation, iterate from O(z^N).)
# 2. Print the first 12 counts (coeffs or labelled, whichever is right).
# 3. Brute force: list the objects for small n by some independent method, and check(...).
# 4. Measure growth with a ratio test on the EGF coefficients F[n+1]/F[n], and on the counts.


raise NotImplementedError("delete this line when the cell is filled in")

**Question 4.2.** Against the output, did each of your three predictions hold? For the growth
prediction, say which number in your output confirms or refutes it. If one missed, find out why:
a missed prediction here is the most interesting thing in your lab, and it should get the most
words.

*Your answer:*

---
# Part 5 · Write-up

Prose, in your own words, for another student in this course. About a page in total.

### 5.1 · What you found, and how you know it

Half a page. Not a narration of what you typed: an account of what is true and how you know it.
For each count you report, say what you checked it against. Say where the O-term told you the
truth and where it did not.

*Your write-up:*

### 5.2 · A prediction that was wrong

Name one prediction from this lab that did not survive contact with the computation, and explain
what you had misunderstood. If every prediction held, say so, and instead name the one you were
least sure of and why. **This section is required and graded as content.**

*Your write-up:*

### 5.3 · Methods and tools

Three or four sentences: what software you used and for what, what you checked and against what,
and any help you had, from a classmate, from me, or from a machine. The course AI policy applies:
help with *syntax* is fine, help deciding *what to compute* is not, and every line you submit
should be one you can explain. All prose in this notebook is your own.

*Your note:*

---
### How it is graded

Holistically, on three things: **predictions made before computing and left visible when wrong;
counts checked against something that did not produce them; and prose that says something.** The
most common way to lose points is a clean notebook with empty prediction cells.

### Before you submit

1. **Kernel → Restart Kernel and Run All Cells.** Read the output top to bottom.
2. No `NotImplementedError` remains.
3. Every prediction cell is filled in, including the ones you got wrong.
4. **Export a PDF**: File → Save and Export Notebook As → PDF (or print the page to PDF). Check
   that the outputs are in it.
5. **Submit both files to Gradescope in one submission**: the `.ipynb` and the `.pdf`. I read and
   mark up the PDF; I run the notebook when I want to check something. Gradescope's mobile app
   cannot submit to this assignment, so use a browser.

---
## Optional · if you want to keep going

Not graded. **Look at a random tree.** A plane tree in which every node independently has $k$
children with probability $2^{-(k+1)}$ is a uniformly random plane tree *given its size* (every plane tree with $n$ nodes
has probability $2^{-(2n-1)}$); this is the simplest case of what is called a Boltzmann sampler.
The cell below draws
such trees until one has between $500$ and $1000$ nodes, then reports its height. Run it a dozen
times and compare the heights with $\sqrt n$. Then plot one with
`T.plot(layout='tree', vertex_size=0, vertex_labels=False)`.

In [ ]:
import random

def random_plane_tree(lo=500, hi=1000):
    "A uniformly random plane tree with between lo and hi nodes (rejection sampling)."
    while True:
        edges, depth, stack, count = [], {0: 0}, [0], 1
        while stack and count <= hi:
            v = stack.pop()
            k = 0
            while random.random() < 0.5:
                k += 1
            for _ in range(k):
                edges.append((v, count)); depth[count] = depth[v] + 1; stack.append(count); count += 1
        if not stack and lo <= count <= hi:
            return Graph(edges), max(depth.values()), count

T, height, n = random_plane_tree()
print("nodes:", n, "  height:", height, "  sqrt(n):", RDF(sqrt(n)))